# Student Earnings Pipeline

### Setup

Download the `data.zip` folder from the google drive and extract it in the project directory.
The directory structure for this pipline should look like the following (place your extracted files into the raw folder):

```md
data/
    raw/
    processed/
data-pipline
    notebooks.ipynb
```
Install the required packages listed in `requirements.txt`

Outputs are saved to `data/processed/`

In [1]:
import polars as pl
import pandas as pd
from pathlib import Path

# Use the current working directory as the starting point for data paths
BASE_DIR = Path.cwd()

RAW_DATA_DIR = BASE_DIR / ".." / "data" / "raw"
PROCESSED_DATA_DIR = BASE_DIR / ".." / "data" / "processed"


# Create the input/ouput folders if they do not already exist
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)
RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)


## Initial Raw Data Processing

In [2]:
def main():
    print("Scanning CSV files...")

    # Class of 2017-2018 only:
    #   - FieldOfStudyData1718_1819_PP.csv: EARN_MDN_HI_1YR = 1 year after graduating
    #   - FieldOfStudyData2122_2223_PP.csv: EARN_MDN_5YR   = 5 years after graduating
    # Output column names (earn_mdn_1yr, earn_mdn_5yr, completions, etc.) are
    # kept identical to the original single-file version -- other scripts
    # depend on this schema -- even though the values now come from two
    # different source files.
    SOURCES = {
        "1yr": {
            "file": "FieldOfStudyData1718_1819_PP.csv",
            "earn_col": "EARN_MDN_HI_1YR",
            "count_col": "EARN_COUNT_WNE_HI_1YR",
        },
        "5yr": {
            "file": "FieldOfStudyData2122_2223_PP.csv",
            "earn_col": "EARN_MDN_5YR",
            "count_col": "EARN_COUNT_WNE_5YR",
        },
    }

    # Scorecard writes literal "NA" in some cols, including UNITID
    nulls = ["NA", "NULL", ""]

    # Helper to turn suppressed text into numbers
    def clean_suppressed(col_name, dtype=pl.Float64):
        return (
            pl.col(col_name)
            .str.strip_chars()
            .replace(["PS", ""], None)
            .cast(dtype, strict=False)
        )

    # Showing better documented CA public universities for demo purposes
    systems = {
        "California State University": "CSU",
        "University of California": "UC",
    }
    ca = (
        pl.read_excel(RAW_DATA_DIR / "CW2024_prelim.xlsx", sheet_name="Crosswalk")
        .select(
            pl.col("IPEDSMatch").cast(pl.Int64, strict=False).alias("UNITID"),
            pl.col("f1sysnam").replace_strict(systems, default=None).alias("system"),
        )
        .drop_nulls()
        .unique(subset="UNITID")
    )

    # Institution metadata
    meta = pl.scan_csv(
        RAW_DATA_DIR / "Most-Recent-Cohorts-Institution.csv",
        schema_overrides={c: pl.String for c in ["UGDS"]},
        null_values=nulls,
        infer_schema_length=10000,
    ).select(
        pl.col("UNITID").cast(pl.Int64),
        pl.col("CITY").alias("city"),
        pl.col("STABBR").alias("state"),
        pl.col("UGDS").cast(pl.Int32, strict=False).alias("undergrad_enrollment"),
    )

    # Build one frame per horizon, each using that file's own column names
    # for the earnings/count/completions measures, renamed to the shared
    # "<measure>_<horizon>" scheme (completions stays per-horizon here and
    # gets coalesced into one column below).
    frames = {}
    for horizon, src in SOURCES.items():
        earn_col, count_col = src["earn_col"], src["count_col"]
        suppressed_cols = [earn_col, count_col, "IPEDSCOUNT1"]

        frame = pl.scan_csv(
            RAW_DATA_DIR / src["file"],
            schema_overrides={c: pl.String for c in suppressed_cols + ["CIPCODE"]},
            null_values=nulls,
            infer_schema_length=10000,
        )

        frame = (
            frame.with_columns(pl.col("UNITID").cast(pl.Int64))
            .join(ca.lazy(), on="UNITID", how="inner")
            .join(meta, on="UNITID", how="left")
            .with_columns(
                # Keep as text, CIP codes can start with a zero (0110)
                pl.col("CIPCODE").str.zfill(4).alias("cipcode"),

                clean_suppressed(earn_col).alias(f"earn_mdn_{horizon}"),
                clean_suppressed(count_col, pl.Int32).alias(f"earn_count_{horizon}"),
                clean_suppressed("IPEDSCOUNT1", pl.Int32).alias(f"completions_{horizon}"),

                # Why a value is missing: "PS" = withheld for privacy, blank = not reported
                pl.when(pl.col(earn_col) == "PS").then(pl.lit("suppressed"))
                  .when(pl.col(earn_col).is_null()).then(pl.lit("not_reported"))
                  .otherwise(pl.lit("usable"))
                  .cast(pl.Categorical).alias(f"status_{horizon}"),
            )
            .select(
                pl.col("UNITID").alias("unitid"),
                pl.col("INSTNM").alias("institution"),
                "system", "city", "state", "undergrad_enrollment",
                "cipcode",
                pl.col("CIPDESC").alias("program"),
                pl.col("CREDLEV").cast(pl.Int16).alias("credlev"),
                f"completions_{horizon}",
                f"earn_mdn_{horizon}",
                f"earn_count_{horizon}",
                f"status_{horizon}",
            )
        )
        frames[horizon] = frame

    join_keys = ["unitid", "cipcode", "credlev"]
    metadata_cols = ["institution", "program", "system", "city", "state", "undergrad_enrollment"]

    df = (
        frames["1yr"]
        .join(frames["5yr"], on=join_keys, how="full", coalesce=True, suffix="_5yr")
        .with_columns([pl.coalesce(pl.col(c), pl.col(f"{c}_5yr")).alias(c) for c in metadata_cols])
        .drop([f"{c}_5yr" for c in metadata_cols])
        .with_columns(pl.coalesce(pl.col("completions_1yr"), pl.col("completions_5yr")).alias("completions"))
        .drop(["completions_1yr", "completions_5yr"])
        .sort(["system", "institution", "cipcode", "credlev"])
    )

    final_df = df.collect()
    final_df.write_parquet(PROCESSED_DATA_DIR / "student-earnings-intermediate.parquet", compression="zstd")

    print(f"Processed {final_df.height:,} records.")
    schema = final_df.schema
    for col_name, dtype in schema.items():
        print(f"{col_name}: {dtype}")


if __name__ == "__main__":
    main()

Could not determine dtype for column 14, falling back to string


Scanning CSV files...
Processed 4,706 records.
unitid: Int64
institution: String
system: String
city: String
state: String
undergrad_enrollment: Int32
cipcode: String
program: String
credlev: Int16
earn_mdn_1yr: Float64
earn_count_1yr: Int32
status_1yr: Categorical
earn_mdn_5yr: Float64
earn_count_5yr: Int32
status_5yr: Categorical
completions: Int32


### Add Column for Inflation Adjusted 1-Year Earnings (2024 Dollars)

In [3]:

# College Scorecard reports 1-year earnings in 2020 dollars and 5-year earnings in 2024 dollars
# We use the Bureau of Labor Statistics (BLS) annual CPI-U averages to account for inflation
# CPI values: 
#   2020 = 258.811
#   2024 = 313.689
# Multiplying by 313.689 / 258.811 (~1.212) converts 1-year earnings to 2024 dollars
# Allows for fairer comparison while preserving the original earnings in a separate column

CPI_2020 = 258.811
CPI_2024 = 313.689

inflation_factor = CPI_2024 / CPI_2020

df = pl.read_parquet(
    PROCESSED_DATA_DIR / "student-earnings-intermediate.parquet"
)

df = df.with_columns(
    (pl.col("earn_mdn_1yr") * inflation_factor)
    .round(2)
    .alias("earn_mdn_1yr_2024")
)

print("Inflation factor:", round(inflation_factor, 6))
print("Original column preserved:", "earn_mdn_1yr" in df.columns)
print("New adjusted column:", "earn_mdn_1yr_2024" in df.columns)

print(
    df.select(
        "earn_mdn_1yr",
        "earn_mdn_1yr_2024",
        "earn_mdn_5yr"
    ).head(5)
)

df.write_parquet(
    PROCESSED_DATA_DIR / "student-earnings-intermediate.parquet",
    compression="zstd"
)

Inflation factor: 1.212039
Original column preserved: True
New adjusted column: True
shape: (5, 3)
┌──────────────┬───────────────────┬──────────────┐
│ earn_mdn_1yr ┆ earn_mdn_1yr_2024 ┆ earn_mdn_5yr │
│ ---          ┆ ---               ┆ ---          │
│ f64          ┆ f64               ┆ f64          │
╞══════════════╪═══════════════════╪══════════════╡
│ 83137.0      ┆ 100765.28         ┆ 131566.0     │
│ null         ┆ null              ┆ null         │
│ 95733.0      ┆ 116032.12         ┆ 124774.0     │
│ null         ┆ null              ┆ null         │
│ null         ┆ null              ┆ null         │
└──────────────┴───────────────────┴──────────────┘


### Assign Departments

In [4]:
earnings = pd.read_parquet(PROCESSED_DATA_DIR / "student-earnings-intermediate.parquet")
buckets = pd.read_csv(RAW_DATA_DIR / "earning_buckets.csv")

# Clean accidental spaces so program names match reliably
earnings["program_key"] = earnings["program"].str.strip()
buckets["program_key"] = buckets["program"].str.strip()

# Keep only the lookup columns, then add department to every earnings row
earnings = earnings.drop(columns="department", errors="ignore")

earnings_with_buckets = earnings.merge(
    buckets[["program_key", "department"]],
    on="program_key",
    how="left",       # keep every row from earnings
    validate="m:1"    # many earnings rows can map to one program bucket
)

earnings_with_buckets = earnings_with_buckets.drop(columns="program_key")

earnings_with_buckets.to_parquet(
    PROCESSED_DATA_DIR / "golden-student-earnings-with-buckets.parquet",
    index=False
)

print("Unmatched rows:", earnings_with_buckets["department"].isna().sum())
print(earnings_with_buckets.columns)

Unmatched rows: 13
Index(['unitid', 'institution', 'system', 'city', 'state',
       'undergrad_enrollment', 'cipcode', 'program', 'credlev', 'earn_mdn_1yr',
       'earn_count_1yr', 'status_1yr', 'earn_mdn_5yr', 'earn_count_5yr',
       'status_5yr', 'completions', 'earn_mdn_1yr_2024', 'department'],
      dtype='str')


### Prepare the Analysis Dataset

- remove columns that are not required for the earnings summaries

In [5]:
needs_cleaning = pd.read_parquet(PROCESSED_DATA_DIR / "golden-student-earnings-with-buckets.parquet")

needs_cleaning = needs_cleaning.drop(columns=[
    "unitid",
    "system",
    "city",
    "state",
    "undergrad_enrollment",
    "cipcode",
    "credlev",
    "completions",
    "status_1yr",
    "status_5yr"
])

### Calculate Earnings Statistics

- shows the mean, median, standard deviation, and 25th and 75th percentiles of program level median earnings by institution and department
- missing earnings are excluded

In [6]:
count_cols = [
    "earn_mdn_1yr",
    "earn_mdn_1yr_2024",
    "earn_mdn_5yr"
]

df = needs_cleaning.copy()

# Convert values to numeric. Keep missing values as NaN:
# Pandas excludes NaN values from mean, median, std, and quantiles.
for col in count_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

grouped = df.groupby(["institution", "department"])


stats_parts = []

for col in count_cols:
    stats = grouped[col].agg(
        mean="mean",
        median="median",
        std="std",
        quantile_25=lambda x: x.quantile(0.25),
        quantile_75=lambda x: x.quantile(0.75),
    )

    # Makes names such as mean_earn_count_1yr
    stats.columns = [f"{stat}_{col}" for stat in stats.columns]
    stats_parts.append(stats)

student_stats = (
    pd.concat(stats_parts, axis=1)
    .reset_index()
    .rename(columns={"institution": "Institution", "department": "department"})
)

student_stats.to_parquet(
    PROCESSED_DATA_DIR / "golden-student-pay-stats.parquet",
    index=False
)

display(student_stats)

,Institution,department,mean_earn_mdn_1yr,median_earn_mdn_1yr,std_earn_mdn_1yr,quantile_25_earn_mdn_1yr,quantile_75_earn_mdn_1yr,mean_earn_mdn_1yr_2024,median_earn_mdn_1yr_2024,std_earn_mdn_1yr_2024,quantile_25_earn_mdn_1yr_2024,quantile_75_earn_mdn_1yr_2024,mean_earn_mdn_5yr,median_earn_mdn_5yr,std_earn_mdn_5yr,quantile_25_earn_mdn_5yr,quantile_75_earn_mdn_5yr
0,Cal Poly Maritime Academy,"Agriculture, Food & Animal",75128.000000,75128.0,NaN,75128.00,75128.00,91058.060000,91058.06,NaN,91058.060,91058.0600,122033.0,122033.0,NaN,122033.00,122033.00
1,Cal Poly Maritime Academy,"Business, Economics & Finance",80071.000000,80071.0,45993.053475,63810.00,96332.00,97049.170000,97049.17,55745.371207,77340.205,116758.1350,108546.5,108546.5,41942.038726,93717.75,123375.25
2,Cal Poly Maritime Academy,Engineering,89435.000000,89435.0,8906.717016,86286.00,92584.00,108398.700000,108398.70,10795.286091,104581.990,112215.4100,128170.0,128170.0,4802.669258,126472.00,129868.00
3,Cal Poly Maritime Academy,Humanities & Languages,40865.000000,40865.0,NaN,40865.00,40865.00,49529.970000,49529.97,NaN,49529.970,49529.9700,90500.0,90500.0,NaN,90500.00,90500.00
4,Cal Poly Maritime Academy,Science & Mathematics,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
321,University of California-Santa Cruz,Engineering,53515.000000,53515.0,NaN,53515.00,53515.00,64862.260000,64862.26,NaN,64862.260,64862.2600,119939.0,119939.0,NaN,119939.00,119939.00
322,University of California-Santa Cruz,Humanities & Languages,27442.909091,28735.0,4726.539865,23936.00,29612.50,33261.875455,34827.94,5728.749445,29011.365,35891.5050,56744.2,56152.5,8134.869524,52469.75,62385.75
323,University of California-Santa Cruz,Medicine & Heath Sciences,23543.000000,23543.0,NaN,23543.00,23543.00,28535.030000,28535.03,NaN,28535.030,28535.0300,69052.0,69052.0,NaN,69052.00,69052.00
324,University of California-Santa Cruz,Science & Mathematics,33747.250000,35188.5,4598.073968,29972.50,36974.75,40902.978750,42649.83,5573.045787,36327.835,44814.8375,82706.0,78785.5,21800.505826,75471.50,84309.00


In [7]:
salaries = pd.read_parquet(PROCESSED_DATA_DIR / "golden-student-earnings-with-buckets.parquet")

salaries.columns


Index(['unitid', 'institution', 'system', 'city', 'state',
       'undergrad_enrollment', 'cipcode', 'program', 'credlev', 'earn_mdn_1yr',
       'earn_count_1yr', 'status_1yr', 'earn_mdn_5yr', 'earn_count_5yr',
       'status_5yr', 'completions', 'earn_mdn_1yr_2024', 'department'],
      dtype='str')